# Boom model training — R50_fines onlyStripped-down version of the full pipeline, predicting only `R50_fines` (median landing distance for fragments < 40mm).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

### Feature helpers

In [ ]:
_LOG_SOURCES = {
    "pi_strength": "log_pi_strength",
    "coupling": "log_coupling",
    "porosity": "log_porosity",
    "shape_factor": "log_shape",
    "energy": "log_energy",
    "strength": "log_strength",
    "gravity": "log_gravity",
    "atmosphere": "log_atmosphere",
}
_EPS = 1e-6

_DERIVED_SOURCES = {
    "log_porosity_sq": ("log_porosity", lambda x: x ** 2),
    "log_coupling_sq": ("log_coupling", lambda x: x ** 2),
    "cos_angle_sq": ("cos_angle", lambda x: x ** 2),
    "log_shape_sq": ("log_shape", lambda x: x ** 2),
    "log_shape_cu": ("log_shape", lambda x: x ** 3),
    "log_energy_cu": ("log_energy", lambda x: x ** 3),
    "log_atm_x_sin": (("log_atmosphere", "sin_angle"), lambda a, b: a * b),
    "log_strength_x_cos": (("log_strength", "cos_angle"), lambda a, b: a * b),
    "log_energy_x_gravity": (("log_energy", "log_gravity"), lambda a, b: a * b),
}


def build_geometry(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()
    X["L_char"] = (X["energy"] / (X["atmosphere"] * X["gravity"])) ** 0.25
    X["pi_strength"] = X["strength"] / (X["atmosphere"] * X["gravity"] * X["L_char"])
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    return X


class ZLogFeatureTransforms:
    def __init__(self):
        self._scalers: dict[str, StandardScaler] = {}

    def _raw_logs(self, geom: pd.DataFrame) -> pd.DataFrame:
        logs = pd.DataFrame(index=geom.index)
        for source, log_col in _LOG_SOURCES.items():
            logs[log_col] = np.log(geom[source].clip(lower=_EPS))
        return logs

    def _compute_derived(self, X: pd.DataFrame, logs: pd.DataFrame) -> dict[str, np.ndarray]:
        raw = {}
        for name, spec in _DERIVED_SOURCES.items():
            src, fn = spec
            if isinstance(src, tuple):
                a = logs[src[0]].to_numpy() if src[0] in logs.columns else X[src[0]].to_numpy()
                b = logs[src[1]].to_numpy() if src[1] in logs.columns else X[src[1]].to_numpy()
                raw[name] = fn(a, b)
            else:
                vals = logs[src].to_numpy() if src in logs.columns else X[src].to_numpy()
                raw[name] = fn(vals)
        return raw

    def fit(self, geom: pd.DataFrame) -> "ZLogFeatureTransforms":
        logs = self._raw_logs(geom)
        for log_col in logs:
            self._scalers[log_col] = StandardScaler().fit(logs[[log_col]].to_numpy())
        X_tmp = geom.copy()
        X_tmp["sin_angle"] = np.sin(X_tmp["angle_rad"])
        X_tmp["cos_angle"] = np.cos(X_tmp["angle_rad"])
        derived = self._compute_derived(X_tmp, logs)
        for name, vals in derived.items():
            self._scalers[name] = StandardScaler().fit(vals.reshape(-1, 1))
        return self

    def transform(self, geom: pd.DataFrame) -> pd.DataFrame:
        X = geom.copy()
        logs = self._raw_logs(geom)
        for log_col in logs:
            raw_log = logs[[log_col]].to_numpy()
            X[f"{log_col}_raw"] = raw_log.ravel()
            X[log_col] = self._scalers[log_col].transform(raw_log).ravel()
        derived = self._compute_derived(X, logs)
        for name, vals in derived.items():
            X[name] = self._scalers[name].transform(vals.reshape(-1, 1)).ravel()
        return X


def build_features(df: pd.DataFrame, z_log: ZLogFeatureTransforms) -> pd.DataFrame:
    return z_log.transform(build_geometry(df))


def transform_target_r50_fines(y: pd.DataFrame, L_char: pd.Series) -> pd.Series:
    return np.log(y["R50_fines"] / L_char)


def invert_target_r50_fines(log_pi_R50_fines: pd.Series, L_char: pd.Series) -> pd.Series:
    return np.exp(log_pi_R50_fines) * L_char

### Model helpers

In [ ]:
LINEAR_FEATURES = [
    "log_pi_strength",
    "log_coupling",
    "log_porosity",
    "log_atmosphere",
    "log_strength",
    "log_shape",
    "sin_angle",
    "cos_angle",
    "log_porosity_sq",
    "log_coupling_sq",
    "cos_angle_sq",
    "log_shape_sq",
    "log_shape_cu",
    "log_energy_cu",
    "log_atm_x_sin",
    "log_strength_x_cos",
    "log_energy_x_gravity",
]

LGB_FEATURES = LINEAR_FEATURES + [
    "log_energy",
    "log_gravity",
]

TARGET = "log_pi_R50_fines"


class HybridPhysicsModel:
    def __init__(self, linear_features, lgb_features, lgb_params=None, ridge_alpha=1.0):
        self.linear_features = linear_features
        self.lgb_features = lgb_features
        self.ridge_alpha = ridge_alpha
        self.lgb_params = lgb_params or {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "min_child_samples": 10,
            "reg_alpha": 0.1,
            "reg_lambda": 0.1,
            "verbose": -1,
        }
        self.baseline = None
        self.residual = None
        self.best_iteration_ = None

    def fit(self, X, y, eval_set=None, eval_names=None, eval_metric="l2", callbacks=None):
        self.baseline = Ridge(alpha=self.ridge_alpha).fit(X[self.linear_features], y)
        residuals = y - self.baseline.predict(X[self.linear_features])
        residual_eval_set = None
        if eval_set is not None:
            residual_eval_set = []
            for X_eval, y_eval in eval_set:
                residual_eval_set.append((
                    X_eval[self.lgb_features],
                    y_eval - self.baseline.predict(X_eval[self.linear_features]),
                ))
        self.residual = lgb.LGBMRegressor(**self.lgb_params).fit(
            X[self.lgb_features],
            residuals,
            eval_set=residual_eval_set,
            eval_names=eval_names,
            eval_metric=eval_metric,
            callbacks=callbacks,
        )
        self.best_iteration_ = getattr(self.residual, "best_iteration_", None)
        return self

    def predict(self, X):
        num_iteration = self.best_iteration_ if self.best_iteration_ else None
        return (
            self.baseline.predict(X[self.linear_features])
            + self.residual.predict(X[self.lgb_features], num_iteration=num_iteration)
        )

    def baseline_coefficients(self):
        return pd.Series(self.baseline.coef_, index=self.linear_features)

### Configuration

In [ ]:
data_dir = "../forward_prediction"
n_splits = 5
seed = 42

### Load raw training data

In [ ]:
X_raw = pd.read_csv(f"{data_dir}/train.csv").reset_index(drop=True)
y_raw = pd.read_csv(f"{data_dir}/train_labels.csv").reset_index(drop=True)
if len(X_raw) != len(y_raw):
    raise ValueError(
        f"{data_dir}/train.csv ({len(X_raw)} rows) and "
        f"{data_dir}/train_labels.csv ({len(y_raw)} rows) must align"
    )

### Feature engineering and transformed target

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

geom = build_geometry(X_raw)
z_log = ZLogFeatureTransforms().fit(geom)
X = build_features(X_raw, z_log)
y = transform_target_r50_fines(y_raw, X["L_char"])

X

In [ ]:
y

### Cross-validation setup

In [ ]:
import os
os.makedirs("../output/R50_fines", exist_ok=True)

kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
cv_rmse = []
cv_mape = []
cv_r2 = []
cv_f1 = []
convergence_history = {}
shap_history = {}
best_iterations = []
early_stopping_rounds = 30
plot_shap_dependence = True

### CV loop: train, validate, log convergence, and early stop

In [ ]:
for fold, (tr, va) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr, y_va = y.iloc[tr], y.iloc[va]
    y_tr_phys = y_raw["R50_fines"].iloc[tr].reset_index(drop=True)

    print(f"\n=== Fold {fold} / {TARGET} convergence ===")
    model = HybridPhysicsModel(linear_features=LINEAR_FEATURES, lgb_features=LGB_FEATURES)

    model.baseline = Ridge(alpha=model.ridge_alpha).fit(X_tr[model.linear_features], y_tr)
    train_residuals = y_tr - model.baseline.predict(X_tr[model.linear_features])
    valid_residuals = y_va - model.baseline.predict(X_va[model.linear_features])

    evals_result = {}
    model.residual = lgb.LGBMRegressor(**model.lgb_params).fit(
        X_tr[model.lgb_features],
        train_residuals,
        eval_set=[
            (X_va[model.lgb_features], valid_residuals),
            (X_tr[model.lgb_features], train_residuals),
        ],
        eval_names=["valid_residual", "train_residual"],
        eval_metric="l2",
        callbacks=[
            lgb.record_evaluation(evals_result),
            lgb.log_evaluation(period=10),
            lgb.early_stopping(stopping_rounds=early_stopping_rounds, first_metric_only=True),
        ],
    )
    model.best_iteration_ = model.residual.best_iteration_
    convergence_history[fold] = evals_result

    valid_l2 = evals_result["valid_residual"]["l2"]
    best_iter = model.best_iteration_ or (int(np.argmin(valid_l2)) + 1)
    best_iterations.append(best_iter)
    print(
        f"Fold {fold}: valid_l2 start={valid_l2[0]:.6g}, "
        f"end={valid_l2[-1]:.6g}, best={min(valid_l2):.6g} at iter {best_iter}"
    )

    preds_transformed = model.predict(X_va)

    explainer = shap.TreeExplainer(model.residual)
    shap_values = explainer.shap_values(X_va[model.lgb_features])
    shap_history[fold] = {
        "values": shap_values,
        "features": X_va[model.lgb_features].copy(),
        "feature_names": model.lgb_features,
    }

    if plot_shap_dependence:
        for fi, feature_to_plot in enumerate(model.lgb_features):
            shap.dependence_plot(
                feature_to_plot,
                shap_values,
                X_va[model.lgb_features],
                interaction_index="auto",
                show=False,
            )
            plt.title(f"SHAP Dependence: {feature_to_plot} (Fold {fold})")
            plt.tight_layout()
            plt.savefig(f"../output/R50_fines/shap_dep_fold{fold}_{fi:02d}.png", dpi=150)
            plt.show()

    preds_physical = invert_target_r50_fines(pd.Series(preds_transformed, index=X_va.index), X_va["L_char"])
    y_va_phys = y_raw["R50_fines"].iloc[va].reset_index(drop=True)
    preds_physical = preds_physical.reset_index(drop=True)

    y_true = y_va_phys.to_numpy()
    y_pred = preds_physical.to_numpy()
    cv_rmse.append(np.sqrt(mean_squared_error(y_true, y_pred)))
    cv_mape.append(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    cv_r2.append(r2_score(y_true, y_pred))

    thr = np.median(y_tr_phys)
    y_bin = (y_true > thr).astype(int)
    p_bin = (y_pred > thr).astype(int)
    cv_f1.append(f1_score(y_bin, p_bin, zero_division=0))

    print(f"Fold {fold} — RMSE={cv_rmse[-1]:.4f}, MAPE={cv_mape[-1]:.2f}%, R²={cv_r2[-1]:.4f}, F1={cv_f1[-1]:.4f}")

### SHAP Summary Plot

In [ ]:
summary_fold = 0
entry = shap_history.get(summary_fold)
if entry is None:
    print(f"No SHAP values found for fold={summary_fold}. Run the CV loop first.")
else:
    shap.summary_plot(entry["values"], entry["features"], show=False)
    plt.title(f"SHAP Summary Plot (Fold {summary_fold}, Target: {TARGET})")
    plt.tight_layout()
    plt.savefig(f"../output/R50_fines/shap_summary.png", dpi=150)
    plt.show()

### LightGBM convergence plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
train_ax = ax.twinx()
for fold in range(n_splits):
    hist = convergence_history.get(fold)
    if hist is None:
        continue
    train_l2 = hist["train_residual"]["l2"]
    valid_l2 = hist["valid_residual"]["l2"]
    iters = np.arange(1, len(valid_l2) + 1)
    color = f"C{fold}"
    ax.plot(iters, valid_l2, color=color, linestyle="-", alpha=0.85, label=f"valid fold {fold}")
    train_ax.plot(iters, train_l2, color=color, linestyle="--", alpha=0.65, label=f"train fold {fold}")
ax.set_title(f"{TARGET} convergence")
ax.set_ylabel("valid residual l2")
ax.set_xlabel("Boosting iteration")
train_ax.set_ylabel("train residual l2")
ax.grid(True, alpha=0.3)
valid_handles, valid_labels = ax.get_legend_handles_labels()
train_handles, train_labels = train_ax.get_legend_handles_labels()
train_ax.legend(valid_handles + train_handles, valid_labels + train_labels, loc="best", fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("../output/R50_fines/lgbm_convergence.png", dpi=150)
print("Saved convergence plot")
plt.show()

### Mean CV metrics (physical units)

In [ ]:
print(f"Mean CV RMSE:  {np.mean(cv_rmse):.4f} ± {np.std(cv_rmse):.4f}")
print(f"Mean CV MAPE:  {np.mean(cv_mape):.2f}% ± {np.std(cv_mape):.2f}%")
print(f"Mean CV R²:    {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"Mean CV F1:    {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")

### CV metric plot

In [ ]:
fold_idx = np.arange(n_splits)
fig, axes = plt.subplots(4, 1, figsize=(8, 10), sharex=True)
ax_rmse, ax_mape, ax_r2, ax_f1 = axes
ax_rmse.plot(fold_idx, cv_rmse, marker="o", label="R50_fines")
ax_mape.plot(fold_idx, cv_mape, marker="o", label="R50_fines")
ax_r2.plot(fold_idx, cv_r2, marker="o", label="R50_fines")
ax_f1.plot(fold_idx, cv_f1, marker="o", label="R50_fines")
ax_rmse.set_ylabel("RMSE"); ax_rmse.set_title("CV RMSE by fold")
ax_mape.set_ylabel("MAPE (%)"); ax_mape.set_title("CV MAPE by fold")
ax_r2.set_ylabel("R²"); ax_r2.set_title("CV R² by fold")
ax_f1.set_xlabel("Fold"); ax_f1.set_ylabel("F1"); ax_f1.set_title("CV F1 by fold (median split)")
for ax in axes:
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
ax_f1.set_xticks(fold_idx)
plt.tight_layout()
plt.savefig("../output/R50_fines/cv_metrics.png", dpi=150)
print("Saved CV metric plot")
plt.show()

### Best iterations from early stopping

In [ ]:
final_n_estimators = int(np.median(best_iterations))
print(f"{TARGET}: folds={best_iterations}, median_for_final_fit={final_n_estimators}")

### Fit final model on full data

In [ ]:
final_model = HybridPhysicsModel(
    linear_features=LINEAR_FEATURES,
    lgb_features=LGB_FEATURES,
    lgb_params={
        "n_estimators": final_n_estimators,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_child_samples": 10,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
        "verbose": -1,
    },
).fit(X, y)
print(f"\n{TARGET} final n_estimators={final_n_estimators} baseline coefficients:")
print(final_model.baseline_coefficients())

### Save trained model

In [ ]:
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/model_R50_fines.joblib")
joblib.dump(z_log, "../models/z_log_R50_fines.joblib")